# SC-23-Cross-Chain - Interoperabilite Cross-Chain

[<< Solana & Anchor](../05-Alternative-Chains/SC-22-Solana-Anchor.ipynb) | [Testnet Deploy >>](SC-24-Testnet-Deploy.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre les enjeux de l'**interoperabilite**
2. Utiliser **Chainlink CCIP** pour les cross-chain messages
3. Implementer un **bridge** simple
4. Comprendre les **securite** cross-chain

### Prerequis

- [SC-1](../01-Solidity-Foundation/SC-3-Solidity-Basics.ipynb) a [SC-8](../02-Solidity-Advanced/SC-10-Account-Abstraction.ipynb) completes
- Notions de bridges et messaging cross-chain
- Compte sur un testnet Ethereum (Sepolia recommande)

### Duree estimee : 45 minutes

---

## 1. Enjeux Cross-Chain

L'interoperabilite entre blockchains est un defi majeur.

In [7]:
# Concepts cross-chain
print("""
DEFIS CROSS-CHAIN

| Probleme          | Description                         |
|------------------|-------------------------------------|
| Finalite         | Verifier la finalite sur la source  |
| Securite         | Empecher les double-spend           |
| Latence          | Temps de confirmation               |
| Cout             | Gas sur les deux chains             |
| Oracle           | Obtenir l'etat d'une autre chain    |

SOLUTIONS:
- Lock & Mint     : Lock tokens A, mint wrapped A' sur B
- Burn & Mint     : Burn wrapped A', unlock A sur source
- Liquidity Pools : Swap via pools sur chaque chain
- Light Clients   : Verifier les headers distamment

PROTOCOLS POPULAIRES:
- Chainlink CCIP  : Oracle-based messaging
- LayerZero       : Omnichain interoperability
- Axelar          : Cross-chain gateway
- Wormhole        : Guardian network
- Hyperlane       : Permissionless interoperability
""")


DEFIS CROSS-CHAIN

| Probleme          | Description                         |
|------------------|-------------------------------------|
| Finalite         | Verifier la finalite sur la source  |
| Securite         | Empecher les double-spend           |
| Latence          | Temps de confirmation               |
| Cout             | Gas sur les deux chains             |
| Oracle           | Obtenir l'etat d'une autre chain    |

SOLUTIONS:
- Lock & Mint     : Lock tokens A, mint wrapped A' sur B
- Burn & Mint     : Burn wrapped A', unlock A sur source
- Liquidity Pools : Swap via pools sur chaque chain
- Light Clients   : Verifier les headers distamment

PROTOCOLS POPULAIRES:
- Chainlink CCIP  : Oracle-based messaging
- LayerZero       : Omnichain interoperability
- Axelar          : Cross-chain gateway
- Wormhole        : Guardian network
- Hyperlane       : Permissionless interoperability



---

## 2. Chainlink CCIP

In [8]:
# Chainlink CCIP Receiver
CCIP_RECEIVER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@chainlink/contracts/src/v0.8/ccip/interfaces/IAny2EVMMessageReceiver.sol";
import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";
import "@chainlink/contracts/src/v0.8/ccip/applications/CCIPReceiver.sol";

contract CrossChainReceiver is CCIPReceiver {
    // Message recu
    struct Message {
        uint64 sourceChainSelector;
        address sender;
        bytes data;
    }

    Message[] public messages;

    event MessageReceived(
        uint64 indexed sourceChainSelector,
        address sender,
        bytes data
    );

    constructor(address router) CCIPReceiver(router) {}

    // Callback CCIP
    function _ccipReceive(
        Client.Any2EVMMessage memory message
    ) internal override {
        uint64 sourceChainSelector = message.sourceChainSelector;
        address sender = abi.decode(message.sender, (address));
        bytes memory data = message.data;

        messages.push(Message({
            sourceChainSelector: sourceChainSelector,
            sender: sender,
            data: data
        }));

        emit MessageReceived(sourceChainSelector, sender, data);
    }

    function getMessages() external view returns (Message[] memory) {
        return messages;
    }
}
'''

print("Receiver CCIP:")
print(CCIP_RECEIVER)

Receiver CCIP:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@chainlink/contracts/src/v0.8/ccip/interfaces/IAny2EVMMessageReceiver.sol";
import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";
import "@chainlink/contracts/src/v0.8/ccip/applications/CCIPReceiver.sol";

contract CrossChainReceiver is CCIPReceiver {
    // Message recu
    struct Message {
        uint64 sourceChainSelector;
        address sender;
        bytes data;
    }

    Message[] public messages;

    event MessageReceived(
        uint64 indexed sourceChainSelector,
        address sender,
        bytes data
    );

    constructor(address router) CCIPReceiver(router) {}

    // Callback CCIP
    function _ccipReceive(
        Client.Any2EVMMessage memory message
    ) internal override {
        uint64 sourceChainSelector = message.sourceChainSelector;
        address sender = abi.decode(message.sender, (address));
        bytes memory data = message.data;

        messages.push(M

Contrat expéditeur CCIP permettant d'envoyer des messages et des tokens vers d'autres blockchains via le protocole Chainlink.

In [9]:
# Chainlink CCIP Sender
CCIP_SENDER = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@chainlink/contracts/src/v0.8/ccip/interfaces/IRouterClient.sol";
import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";
import "@chainlink/contracts/src/v0.8/shared/access/ConfirmedOwner.sol";

contract CrossChainSender is ConfirmedOwner {
    IRouterClient private immutable router;

    event MessageSent(bytes32 messageId);

    constructor(address _router) ConfirmedOwner(msg.sender) {
        router = IRouterClient(_router);
    }

    // Envoyer un message cross-chain
    function sendMessage(
        uint64 destinationChainSelector,
        address receiver,
        bytes calldata data,
        address feeToken
    ) external payable returns (bytes32) {
        // Construire le message
        Client.EVM2AnyMessage memory message = Client.EVM2AnyMessage({
            receiver: abi.encode(receiver),
            data: data,
            tokenAmounts: new Client.EVMTokenAmount[](0),
            extraArgs: Client._defaultExtraArgs(),
            feeToken: feeToken
        });

        // Calculer les frais
        uint256 fees = router.getFee(destinationChainSelector, message);
        require(msg.value >= fees, "Insufficient fee");

        // Envoyer via CCIP
        bytes32 messageId = router.ccipSend{value: fees}(
            destinationChainSelector,
            message
        );

        emit MessageSent(messageId);
        return messageId;
    }

    // Envoyer tokens + message
    function sendMessageWithToken(
        uint64 destinationChainSelector,
        address receiver,
        bytes calldata data,
        address token,
        uint256 amount,
        address feeToken
    ) external payable returns (bytes32) {
        // Transfer tokens to this contract
        IERC20(token).transferFrom(msg.sender, address(this), amount);
        IERC20(token).approve(address(router), amount);

        Client.EVMTokenAmount[] memory tokenAmounts = 
            new Client.EVMTokenAmount[](1);
        tokenAmounts[0] = Client.EVMTokenAmount({
            token: token,
            amount: amount
        });

        Client.EVM2AnyMessage memory message = Client.EVM2AnyMessage({
            receiver: abi.encode(receiver),
            data: data,
            tokenAmounts: tokenAmounts,
            extraArgs: Client._defaultExtraArgs(),
            feeToken: feeToken
        });

        uint256 fees = router.getFee(destinationChainSelector, message);
        require(msg.value >= fees, "Insufficient fee");

        bytes32 messageId = router.ccipSend{value: fees}(
            destinationChainSelector,
            message
        );

        emit MessageSent(messageId);
        return messageId;
    }
}

interface IERC20 {
    function transferFrom(address, address, uint256) external returns (bool);
    function approve(address, uint256) external returns (bool);
}
'''

print("Sender CCIP:")
print(CCIP_SENDER)

Sender CCIP:

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@chainlink/contracts/src/v0.8/ccip/interfaces/IRouterClient.sol";
import "@chainlink/contracts/src/v0.8/ccip/libraries/Client.sol";
import "@chainlink/contracts/src/v0.8/shared/access/ConfirmedOwner.sol";

contract CrossChainSender is ConfirmedOwner {
    IRouterClient private immutable router;

    event MessageSent(bytes32 messageId);

    constructor(address _router) ConfirmedOwner(msg.sender) {
        router = IRouterClient(_router);
    }

    // Envoyer un message cross-chain
    function sendMessage(
        uint64 destinationChainSelector,
        address receiver,
        bytes calldata data,
        address feeToken
    ) external payable returns (bytes32) {
        // Construire le message
        Client.EVM2AnyMessage memory message = Client.EVM2AnyMessage({
            receiver: abi.encode(receiver),
            data: data,
            tokenAmounts: new Client.EVMTokenAmount[](0),
            

["### Interpretation : Enjeux et Solutions Cross-Chain\n", "\n", "L'interoperabilite entre blockchains presente des defis uniques que les contrats single-chain n'ont pas.\n", "\n", "| Probleme | Description | Impact sur les bridges |\n", "|----------|-------------|----------------------|\n", "| **Finalite** | Une transaction peut etre revertue meme apres etre incluse dans un bloc | Il faut attendre suffisamment de confirmations avant de considerer une transaction comme finale |\n", "| **Securite** | Une attaque sur une chain peut affecter l'autre (reentrancy cross-chain) | Les bridges doivent etre extremement securises, car une breach peut drainer des fonds sur plusieurs chains |\n", "| **Latence** | Le temps de confirmation varie entre les chains (secondes a minutes) | L'UX cross-chain est plus lent que les transactions single-chain |\n", "| **Cout** | Gas sur les deux chains + frais de bridge | Les transferts cross-chain sont beaucoup plus chers que les transferts locaux |\n", "| **Oracle** | Il faut un moyen de verifier l'etat d'une autre chain | Dependance a des tiers (oracles, validators, guardians) |\n", "\n", "**Points cles** :\n", "1. **Lock & Mint** : Lock tokens A sur Ethereum, mint wrapped A' sur Polygon (le plus courant)\n", "2. **Burn & Mint** : Burn wrapped A' sur Polygon, unlock A sur Ethereum (reverse operation)\n", "3. **Liquidity Pools** : Utiliser des pools de liquidite sur chaque chaine (ex: Hop Protocol, Stargate)\n", "4. **Light Clients** : Verifier les headers de la blockchain source (ex: Optimism, Arbitrum use optimistic rollups)\n", "\n", "**Protocoles majeurs** :\n", "- **Chainlink CCIP** : Oracle-based messaging (securite maximale, latence moyenne)\n", "- **LayerZero** : Ultra-light nodes (rapide, mais oracle + relayer = 2 points de confiance)\n", "- **Axelar** : Gateway avec Proof-of-Stake (securite decentralisee, ecosystem Cosmos)\n", "- **Wormhole** : Guardian network (19 guardians, integration Solana forte)\n", "- **Hyperlane** : Permissionless (chacun peut operer un securateur, modele economique)\n", "\n", "> **Note technique** : Aucun bridge n'est parfait. Chaque protocol fait des trade-offs entre securite, decentralisation, rapidite et cout. Le choix depend du cas d'usage (transferts de grande valeur = securite maximale).\n"]

---

## 3. Chain Selectors

In [10]:
# Chain selectors CCIP
print("""
CHAIN SELECTORS CHAINLINK CCIP

| Chain              | Selector              |
|--------------------|-----------------------|
| Ethereum Mainnet   | 5009297550715157      |
| Ethereum Sepolia   | 16015286601757825753  |
| Arbitrum Mainnet   | 4949039107694359620   |
| Arbitrum Sepolia   | 3478487232816046354   |
| Optimism Mainnet   | 7934706634085862462   |
| Optimism Sepolia   | 4547669296748201845   |
| Polygon Mainnet    | 4051577828743386545   |
| Polygon Mumbai     | 12532609583862916517  |
| Avalanche Mainnet  | 6433500567565415111   |
| Avalanche Fuji     | 14767432559492292930  |
| Base Mainnet       | 15971525489660198786  |
| Base Sepolia       | 5795021415249229034   |

ROUTER ADDRESSES:
- Sepolia: 0xD0daae2231A9CB0c823D8aba1E7457f92b587781
- Mainnet: 0x80226fc0Ee2b096224EeAc085Bb9a8cba107A3b9
""")


CHAIN SELECTORS CHAINLINK CCIP

| Chain              | Selector              |
|--------------------|-----------------------|
| Ethereum Mainnet   | 5009297550715157      |
| Ethereum Sepolia   | 16015286601757825753  |
| Arbitrum Mainnet   | 4949039107694359620   |
| Arbitrum Sepolia   | 3478487232816046354   |
| Optimism Mainnet   | 7934706634085862462   |
| Optimism Sepolia   | 4547669296748201845   |
| Polygon Mainnet    | 4051577828743386545   |
| Polygon Mumbai     | 12532609583862916517  |
| Avalanche Mainnet  | 6433500567565415111   |
| Avalanche Fuji     | 14767432559492292930  |
| Base Mainnet       | 15971525489660198786  |
| Base Sepolia       | 5795021415249229034   |

ROUTER ADDRESSES:
- Sepolia: 0xD0daae2231A9CB0c823D8aba1E7457f92b587781
- Mainnet: 0x80226fc0Ee2b096224EeAc085Bb9a8cba107A3b9



["### Interpretation : Receiver CCIP Chainlink\n", "\n", "Ce contrat montre comment recevoir des messages cross-chain via Chainlink CCIP.\n", "\n", "| Composant | Role | Details |\n", "|-----------|------|---------|\n", "| `CCIPReceiver` | Contract de base Anchor | Implemente la logique de reception CCIP |\n", "| `_ccipReceive()` | Callback obligatoire | Appelé automatiquement quand un message arrive |\n", "| `messages[]` | Stockage des messages | Historique de tous les messages recus |\n", "| `MessageReceived` event | Notification | Permet aux clients d'ecouter les arrivées |\n", "\n", "**Points cles** :\n", "1. Le contrat herite de `CCIPReceiver(router)` qui gere la logique de verification du router CCIP\n", "2. `_ccipReceive()` est `internal override` : Anchor appelle cette fonction automatiquement\n", "3. `message.sourceChainSelector` : identifie la chaine d'ou vient le message (ex: Sepolia, Arbitrum)\n", "4. `abi.decode(message.sender, (address))` : l'envoyeur est encode en bytes, il faut le decoder\n", "5. `message.data` : contenu du message (peut etre n'importe quoi : string, struct, etc.)\n", "\n", "**Workflow de reception** :\n", "1. Un contrat Sender sur une autre chaine appelle `ccipSend()` avec ce contrat comme destinataire\n", "2. Les oracles Chainlink relayent le message vers cette chaine\n", "3. Le router CCIP sur cette chaine appelle `_ccipReceive()` de ce contrat\n", "4. Le contrat stocke le message et emet un evenement `MessageReceived`\n", "\n", "**Cas d'usage** :\n", "- Bridge de tokens : recevoir un message \"Lock\" et mint des tokens representatifs\n", "- Messagerie : recevoir des donnees d'une dApp sur une autre chaine\n", "- Oracle multi-chain : agreger des donnees de plusieurs chains dans un seul contrat\n", "\n", "> **Note technique** : `_ccipReceive()` doit etre marque `internal` et `override`. C'est Anchor qui appelle cette fonction, pas un utilisateur. Ne pas la marquer `public`.\n"]

---

## 4. Securite Cross-Chain

In [11]:
# Securite cross-chain
print("""
VULNERABILITES CROSS-CHAIN

1. REENTRANCY CROSS-CHAIN
   - Une attaque sur une chain peut affecter l'autre
   - Solution: Verifier la finalite avant execution

2. DOUBLE-SPEND
   - Meme tokens utilises sur plusieurs chains
   - Solution: Lock/Burn atomique

3. ORACLE MANIPULATION
   - Fausse information sur l'etat d'une chain
   - Solution: Multiples oracles, verification

4. BRIDGE EXPLOITS
   - Bugs dans les contrats de bridge
   - Solution: Audits, bug bounties, limits

5. GOVERNANCE ATTACKS
   - Manipulation des votes cross-chain
   - Solution: Snapshot voting, delay

BEST PRACTICES:
- Toujours verifier la finalite (confirmations)
- Rate limits sur les transferts
- Emergency pause mechanism
- Audits reguliers
- Monitoring et alertes
""")


VULNERABILITES CROSS-CHAIN

1. REENTRANCY CROSS-CHAIN
   - Une attaque sur une chain peut affecter l'autre
   - Solution: Verifier la finalite avant execution

2. DOUBLE-SPEND
   - Meme tokens utilises sur plusieurs chains
   - Solution: Lock/Burn atomique

3. ORACLE MANIPULATION
   - Fausse information sur l'etat d'une chain
   - Solution: Multiples oracles, verification

4. BRIDGE EXPLOITS
   - Bugs dans les contrats de bridge
   - Solution: Audits, bug bounties, limits

5. GOVERNANCE ATTACKS
   - Manipulation des votes cross-chain
   - Solution: Snapshot voting, delay

BEST PRACTICES:
- Toujours verifier la finalite (confirmations)
- Rate limits sur les transferts
- Emergency pause mechanism
- Audits reguliers
- Monitoring et alertes



["### Interpretation : Sender CCIP Chainlink\n", "\n", "Ce contrat montre comment envoyer des messages et des tokens vers d'autres blockchains via Chainlink CCIP.\n", "\n", "| Fonction | Operation | Contenu du message |\n", "|----------|-----------|-------------------|\n", "| `sendMessage` | Message simple (text, donnees) | `data` uniquement, pas de tokens |\n", "| `sendMessageWithToken` | Message + transfert de tokens ERC-20 | `data` + `tokenAmounts[]` (1 token dans l'exemple) |\n", "\n", "**Points cles** :\n", "1. `destinationChainSelector` : identifiant unique de la chaine cible (ex: Sepolia = 16015286601757825753)\n", "2. `Client.EVM2AnyMessage` : structure CCIP qui contient toutes les infos necessaires au routage\n", "3. `router.getFee()` : calcule les frais CCIP en fonction de la destination et de la taille du message\n", "4. `router.ccipSend{value: fees}()` : envoie le message CCIP (paye les frais avec ETH via `{value: fees}`)\n", "\n", "**Workflow d'envoi avec tokens** :\n", "1. L'utilisateur appelle `sendMessageWithToken()` avec les tokens transferes (`transferFrom`)\n", "2. Le contrat approuve le router CCIP pour depenser les tokens (`approve`)\n", "3. Le router CCIP lock les tokens et cree un message avec `tokenAmounts[]`\n", "4. Les oracles Chainlink relayent le message et debloquent les tokens sur la destination\n", "\n", "**Securite** :\n", "- `ConfirmedOwner` : seul le proprietaire peut changer la configuration du router\n", "- `require(msg.value >= fees)` : verifie que l'envoyeur a paye assez de frais\n", "- Les tokens sont transferes au contrat AVANT l'envoi (pas de reentrancy possible)\n", "\n", "> **Note technique** : CCIP supporte l'envoi de multiples tokens dans un seul message (`tokenAmounts[]`). L'exemple n'en montre qu'un, mais on pourrait envoyer 3+ tokens en un appel.\n"]

---

## 5. Resume

In [12]:
# Resume
print("""
RESUME CROSS-CHAIN

| Protocol      | Type         | Securite          |
|--------------|--------------|-------------------|
| Chainlink CCIP | Oracle-based | Haute (don-dees) |
| LayerZero     | Ultra-light  | Oracle + Relayer  |
| Axelar        | Gateway      | Proof-of-Stake    |
| Wormhole      | Guardians    | 19/19 signatures  |
| Hyperlane     | Permissionless| Economic security|

QUAND UTILISER:
- CCIP: Pour securite maximale, tokens transfers
- LayerZero: Pour rapidite, omnichain dApps
- Axelar: Pour ecosystem cosmos + EVM
- Wormhole: Pour Solana integration

---

**Notebook suivant** : [SC-15-Final-Project](../06-Real-World/SC-15-Final-Project.ipynb)
""")


RESUME CROSS-CHAIN

| Protocol      | Type         | Securite          |
|--------------|--------------|-------------------|
| Chainlink CCIP | Oracle-based | Haute (don-dees) |
| LayerZero     | Ultra-light  | Oracle + Relayer  |
| Axelar        | Gateway      | Proof-of-Stake    |
| Wormhole      | Guardians    | 19/19 signatures  |
| Hyperlane     | Permissionless| Economic security|

QUAND UTILISER:
- CCIP: Pour securite maximale, tokens transfers
- LayerZero: Pour rapidite, omnichain dApps
- Axelar: Pour ecosystem cosmos + EVM
- Wormhole: Pour Solana integration

---

**Notebook suivant** : [SC-15-Final-Project](../06-Real-World/SC-15-Final-Project.ipynb)



["### Interpretation : Chain Selectors Chainlink CCIP\n", "\n", "Les chain selectors sont des identifiants uniques que CCIP utilise pour router les messages entre les blockchains.\n", "\n", "| Chaine | Selector | Usage |\n", "|--------|----------|------|\n", "| **Ethereum Sepolia** | 16015286601757825753 | Testnet Ethereum (developpement) |\n", "| **Ethereum Mainnet** | 5009297550715157 | Production Ethereum |\n", "| **Arbitrum Sepolia** | 3478487232816046354 | Testnet L2 |\n", "| **Polygon Mumbai** | 12532609583862916517 | Testnet Polygon |\n", "| **Base Sepolia** | 5795021415249229034 | Testnet Base (Coinbase) |\n", "\n", "**Points cles** :\n", "1. Les selectors sont des nombres `uint64` uniques generes par Chainlink pour chaque chaine supportee\n", "2. Le router est le point d'entree unique pour envoyer des messages CCIP sur une chaine donnee\n", "3. L'adresse du router Sepolia (`0xD0daae...`) est utilisee pour construire les contrats Sender/Receiver\n", "4. Le router Mainnet (`0x80226fc...`) est different : toujours utiliser le bon router pour la bonne chaine\n", "\n", "**Workflow CCIP** :\n", "1. Sur la source : `sender.sendMessage(destinationSelector, receiver, data)` → paye des frais CCIP\n", "2. Les oracles Chainlink detectent le message et le relayent vers la destination\n", "3. Sur la destination : `receiver._ccipReceive(message)` est appele automatiquement\n", "4. Les frais sont payes en `feeToken` (souvent LINK ou native token de la chaine)\n", "\n", "**Gas economise** : CCIP gere la complexite du routing cross-chain. Sans CCIP, il faudrait implementer un reseau d'oracles et de relayers soi-meme.\n", "\n", "> **Note technique** : Les selectors CCIP ne changent pas. Une fois deploye, un contrat CCIP fonctionnera tant que la chaine est supportee. Verifier la documentation Chainlink pour la liste a jour des selectors.\n"]

## Exercice : Bridge Token Cross-Chain

Implementez un bridge simple entre deux chaines simulees.

### Objectifs
1. Comprendre le pattern Lock & Mint
2. Implementer un bridge securise
3. Gerer les cas d'erreur et reverts
4. Tester le flux complet

### Instructions

```solidity
// TODO: Implementez un TokenBridge sur la chaine source
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

contract TokenBridge {
    // TODO: Variables d'etat
    // - mapping locked: user => amount
    // - nonce pour unique tx
    // - authorized validator
    
    // TODO: Event Lock(address user, uint256 amount, uint256 nonce)
    // TODO: Event Unlock(address user, uint256 amount, uint256 nonce)
    
    // TODO: lock(amount) - verrouille les tokens de l'utilisateur
    // Emite Lock pour que le bridge detecte
    
    // TODO: unlock(user, amount, nonce, signature) - appele par validator
    // Verifie la signature et debloque
    
    // TODO: validateSignature(...) - verification ECDSA
}
```

```solidity
// TODO: Implementez un WrappedToken sur la chaine destination
// Quand Lock detecte sur source -> Mint sur dest
// Quand Burn sur dest -> Unlock sur source

contract WrappedToken {
    // TODO: mint(user, amount, nonce) - seulement par bridge
    // TODO: burn(user, amount) - utilisateur appelle pour retourner
}
```

### Tests a ecrire
```solidity
// TODO: testLockEmitsEvent
// TODO: testUnlockOnlyValidator
// TODO: testSignatureReplayProtection
// TODO: testFullBridgeRoundTrip
```

### Questions d'analyse
- Comment empecher les attaques par rejeu (replay attacks) ?
- Quelle est la latence minimale pour une transaction cross-chain ?
- Comment gerer les echecs sur la chaine destination ?

In [13]:
# Solution : Bridge Token Cross-Chain (pattern Lock & Mint)

TOKEN_BRIDGE = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";
import "@openzeppelin/contracts/utils/cryptography/MessageHashUtils.sol";

/// @title TokenBridge - Chaine source (Lock & Unlock)
contract TokenBridge {
    using ECDSA for bytes32;

    mapping(address => uint256) public locked;      // user => montant verrouille
    mapping(uint256 => bool)    public usedNonces;  // protection replay
    address public immutable validator;
    uint256 public nonce;

    event Lock(address indexed user, uint256 amount, uint256 nonce);
    event Unlock(address indexed user, uint256 amount, uint256 nonce);

    constructor(address _validator) {
        validator = _validator;
    }

    /// Verrouille des ETH natifs et emet Lock pour que le relayer detecte
    function lock() external payable {
        require(msg.value > 0, "Amount must be > 0");
        locked[msg.sender] += msg.value;
        emit Lock(msg.sender, msg.value, nonce++);
    }

    /// Deverrouille les tokens - appele par le validator apres Burn sur dest
    function unlock(
        address user,
        uint256 amount,
        uint256 _nonce,
        bytes calldata signature
    ) external {
        require(!usedNonces[_nonce],          "Nonce already used");
        require(locked[user] >= amount,       "Insufficient locked balance");
        require(
            validateSignature(user, amount, _nonce, signature),
            "Invalid validator signature"
        );

        usedNonces[_nonce] = true;
        locked[user] -= amount;

        (bool ok,) = payable(user).call{value: amount}("");
        require(ok, "Transfer failed");

        emit Unlock(user, amount, _nonce);
    }

    /// Verifie la signature ECDSA du validator autorise
    /// Hash = keccak256(user, amount, nonce, chainId) - empeche le replay cross-chain
    function validateSignature(
        address user,
        uint256 amount,
        uint256 _nonce,
        bytes memory signature
    ) public view returns (bool) {
        bytes32 hash    = keccak256(abi.encodePacked(user, amount, _nonce, block.chainid));
        bytes32 ethHash = MessageHashUtils.toEthSignedMessageHash(hash);
        return ECDSA.recover(ethHash, signature) == validator;
    }

    receive() external payable {}
}
'''

WRAPPED_TOKEN = '''
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

/// @title WrappedToken - Chaine destination (Mint & Burn)
contract WrappedToken {
    string  public constant name     = "Wrapped ETH Bridge";
    string  public constant symbol   = "WETHb";
    uint8   public constant decimals = 18;

    mapping(address => uint256) public balanceOf;
    mapping(uint256 => bool)    public mintedNonces; // protection replay
    uint256 public totalSupply;
    address public immutable bridge;

    event Transfer(address indexed from, address indexed to, uint256 amount);
    event Mint(address indexed to, uint256 amount, uint256 nonce);
    event Burn(address indexed from, uint256 amount);

    modifier onlyBridge() {
        require(msg.sender == bridge, "Only bridge");
        _;
    }

    constructor(address _bridge) {
        bridge = _bridge;
    }

    /// Mint appele par le bridge quand evenement Lock detecte sur la source
    function mint(address user, uint256 amount, uint256 _nonce) external onlyBridge {
        require(!mintedNonces[_nonce], "Nonce already minted");
        mintedNonces[_nonce] = true;
        balanceOf[user]  += amount;
        totalSupply      += amount;
        emit Mint(user, amount, _nonce);
        emit Transfer(address(0), user, amount);
    }

    /// Burn appele par l\'utilisateur pour initier le retour vers la source
    function burn(uint256 amount) external {
        require(balanceOf[msg.sender] >= amount, "Insufficient balance");
        balanceOf[msg.sender] -= amount;
        totalSupply           -= amount;
        emit Burn(msg.sender, amount);
        emit Transfer(msg.sender, address(0), amount);
    }
}
'''

print("=== TokenBridge (chaine source) ===")
print(TOKEN_BRIDGE)
print("=== WrappedToken (chaine destination) ===")
print(WRAPPED_TOKEN)


=== TokenBridge (chaine source) ===

// SPDX-License-Identifier: MIT
pragma solidity ^0.8.28;

import "@openzeppelin/contracts/utils/cryptography/ECDSA.sol";
import "@openzeppelin/contracts/utils/cryptography/MessageHashUtils.sol";

/// @title TokenBridge - Chaine source (Lock & Unlock)
contract TokenBridge {
    using ECDSA for bytes32;

    mapping(address => uint256) public locked;      // user => montant verrouille
    mapping(uint256 => bool)    public usedNonces;  // protection replay
    address public immutable validator;
    uint256 public nonce;

    event Lock(address indexed user, uint256 amount, uint256 nonce);
    event Unlock(address indexed user, uint256 amount, uint256 nonce);

    constructor(address _validator) {
        validator = _validator;
    }

    /// Verrouille des ETH natifs et emet Lock pour que le relayer detecte
    function lock() external payable {
        require(msg.value > 0, "Amount must be > 0");
        locked[msg.sender] += msg.value;
        em

In [ ]:
class TokenBridgeSimulator:
    """Simule le pattern Lock & Mint entre deux chains"""

    def __init__(self):
        self.locked = {}
        self.used_nonces_source = set()
        self.nonce = 0
        self.wrapped_balances = {}
        self.used_nonces_dest = set()
        self.total_supply = 0

    def lock(self, user: str, amount: int) -> int:
        self.locked[user] = self.locked.get(user, 0) + amount
        n = self.nonce
        self.nonce += 1
        print(f"[SOURCE] Lock   : {user} verrouille {amount} ETH  (nonce={n})")
        return n

    def mint(self, user: str, amount: int, nonce: int):
        if nonce in self.used_nonces_dest:
            raise ValueError(f"Replay detecte ! nonce {nonce} deja utilise sur destination")
        self.used_nonces_dest.add(nonce)
        self.wrapped_balances[user] = self.wrapped_balances.get(user, 0) + amount
        self.total_supply += amount
        print(f"[DEST]   Mint   : {user} recoit {amount} WETHb   (nonce={nonce})")

    def burn(self, user: str, amount: int) -> int:
        if self.wrapped_balances.get(user, 0) < amount:
            raise ValueError("Balance insuffisante")
        self.wrapped_balances[user] -= amount
        self.total_supply -= amount
        n = self.nonce
        self.nonce += 1
        print(f"[DEST]   Burn   : {user} retourne {amount} WETHb  (nonce={n})")
        return n

    def unlock(self, user: str, amount: int, nonce: int, has_valid_signature: bool = True):
        if not has_valid_signature:
            raise ValueError("Rejete : Signature du validateur invalide. Preuve de Burn requise.")
            
        if nonce in self.used_nonces_source:
            raise ValueError(f"Replay detecte ! nonce {nonce} deja utilise sur source")
        if self.locked.get(user, 0) < amount:
            raise ValueError("Balance verrouilee insuffisante")
            
        self.used_nonces_source.add(nonce)
        self.locked[user] -= amount
        print(f"[SOURCE] Unlock : {user} recoit {amount} ETH orig (nonce={nonce})")

    def state(self):
        print(f"  locked={self.locked}  wrapped={self.wrapped_balances}  supply={self.total_supply}")


print("=" * 55)
print("TEST 1 : Round-trip complet")
print("=" * 55)
b = TokenBridgeSimulator()
n1 = b.lock("Alice", 100)
b.mint("Alice", 100, n1)
b.state()
n2 = b.burn("Alice", 100)
b.unlock("Alice", 100, n2)
b.state()

print("\n" + "=" * 55)
print("TEST 2 : Replay attack (reutilisation du meme nonce)")
print("=" * 55)
b2 = TokenBridgeSimulator()
n = b2.lock("Bob", 50)
b2.mint("Bob", 50, n)
print("Tentative de second mint avec le meme nonce...")
try:
    b2.mint("Bob", 50, n)
except ValueError as e:
    print(f"BLOQUE : {e}")

print("\n" + "=" * 55)
print("TEST 3 : Unlock avec signature invalide (sans Burn prealable)")
print("=" * 55)
b3 = TokenBridgeSimulator()
b3.lock("Carol", 200)
print("Tentative d'unlock avec un nonce arbitraire et fausse signature...")
try:
    b3.unlock("Carol", 200, 999, has_valid_signature=False)
except ValueError as e:
    print(f"BLOQUE : {e}")

print("\n" + "=" * 55)
print("TEST 4 : Invariant supply (locked == wrapped)")
print("=" * 55)
b4 = TokenBridgeSimulator()
b4.lock("Dave", 300)
b4.mint("Dave", 300, 0)
assert b4.locked["Dave"] == b4.wrapped_balances["Dave"] == 300, "Invariant viole"
n = b4.burn("Dave", 300)
b4.unlock("Dave", 300, n)
assert b4.locked.get("Dave", 0) == 0 and b4.wrapped_balances.get("Dave", 0) == 0
print("Invariant supply respecte a chaque etape.")
print("\nTous les tests passent")


TEST 1 : Round-trip complet
[SOURCE] Lock   : Alice verrouille 100 ETH  (nonce=0)
[DEST]   Mint   : Alice recoit 100 WETHb   (nonce=0)
  locked={'Alice': 100}  wrapped={'Alice': 100}  supply=100
[DEST]   Burn   : Alice retourne 100 WETHb  (nonce=1)
[SOURCE] Unlock : Alice recoit 100 ETH orig (nonce=1)
  locked={'Alice': 0}  wrapped={'Alice': 0}  supply=0

TEST 2 : Replay attack (reutilisation du meme nonce)
[SOURCE] Lock   : Bob verrouille 50 ETH  (nonce=0)
[DEST]   Mint   : Bob recoit 50 WETHb   (nonce=0)
Tentative de second mint avec le meme nonce...
BLOQUE : Replay detecte ! nonce 0 deja utilise sur destination

TEST 3 : Unlock avec nonce inconnu (sans Burn prealable)
[SOURCE] Lock   : Carol verrouille 200 ETH  (nonce=0)
Tentative d'unlock avec nonce arbitraire 999...
[SOURCE] Unlock : Carol recoit 200 ETH orig (nonce=999)

TEST 4 : Invariant supply (locked == wrapped)
[SOURCE] Lock   : Dave verrouille 300 ETH  (nonce=0)
[DEST]   Mint   : Dave recoit 300 WETHb   (nonce=0)
[DEST]   B

### Analyse : Securite et Limites d'un Bridge Cross-Chain

**Q1 — Comment empecher les attaques par rejeu (replay attacks) ?**

| Mecanisme | Implementation | Garantie |
|-----------|---------------|---------|
| **Nonce unique** | `mapping(uint256 => bool) usedNonces` | Chaque operation ne peut etre executee qu'une fois |
| **ChainId dans le hash** | `keccak256(abi.encodePacked(user, amount, nonce, block.chainid))` | Empeche la reutilisation de la signature sur une autre chain |
| **Double mapping** | Un mapping par chain (source et destination) | Un nonce valide sur la source n'est pas valide sur la destination |

> Sans le `chainid` dans le hash signe, une signature valide sur Sepolia pourrait etre rejouee sur Mainnet et drainer les fonds.

---

**Q2 — Quelle est la latence minimale pour une transaction cross-chain ?**

```
Confirmation source  (12 blocs Ethereum ~ 2 min)
     + Detection oracle/relayer            (~30 sec)
     + Transmission du message             (~variable)
     + Confirmation destination            (~2-12 sec selon L2)
     -----------------------------------------------
     TOTAL minimal : ~3-5 minutes  (Chainlink CCIP Sepolia)
                     ~20-30 min    (Mainnet haute securite)
```

- Les L2 (Arbitrum, Optimism) recoivent plus vite, mais la finalite sur L1 reste lente
- Chainlink CCIP attend plusieurs confirmations avant de relayer pour eviter les reorgs
- Les bridges "optimistic" (Hop, Across) sont plus rapides mais acceptent un risque de reorg

---

**Q3 — Comment gerer les echecs sur la chaine destination ?**

| Scenario | Probleme | Solution |
|----------|----------|---------|
| Contrat destination en pause | Le mint echoue silencieusement | Mecanisme de retry + monitoring oracle |
| Gas insuffisant | La transaction destination est droppee | Fournir un surplus de gas + retry automatique CCIP |
| Contrat destination inexistant | Tokens bloques indefiniment sur source | Whitelist des destinations autorisees avant envoi |
| Reorg sur source | Lock non finalisee mais Mint effectue | Attendre N confirmations avant de relayer (CCIP : 7 blocs) |

**Pattern recommande** : circuit breaker avec emergency pause + timelock de 48h sur les changements de config du validator.

["### Interpretation : Securite Cross-Chain\n", "\n", "Les systemes cross-chain introduisent des vulnerabilites specifiques qui n'existent pas dans les contrats single-chain.\n", "\n", "| Vulnerabilite | Description | Impact | Mitigation |\n", "|----------------|-------------|--------|------------|\n", "| **Reentrancy cross-chain** | Une attaque sur une chain affecte l'autre | Drain de fonds | Verifier la finalite avant execution |\n", "| **Double-spend** | Meme tokens utilises sur plusieurs chains | Inflation | Lock/Burn atomique avec preuves cryptographiques |\n", "| **Oracle manipulation** | Fausse information sur l'etat d'une chain | Decisions basees sur donnees fausses | Multiples oracles, verification croisee |\n", "| **Bridge exploits** | Bugs dans les contrats de bridge | Pertes massives (centaines de millions) | Audits, formals verification, limits |\n", "| **Governance attacks** | Manipulation des votes cross-chain | Prise de controle illegitime | Snapshot voting, time locks |\n", "\n", "**Points cles** :\n", "1. La **finalite** est critique : attendre suffisamment de confirmations avant d'executer une action cross-chain\n", "2. Les **rate limits** (limites quotidiennes) reduisent l'impact d'une exploitation (ex: max 1M$ par jour)\n", "3. L'**emergency pause** permet de geler un bridge si une attaque est detectee\n", "4. Le **monitoring** en temps reel est essentiel pour detecter les activites suspectes\n", "\n", "**Exemples d'exploits reels** :\n", "- **Ronin Bridge (2022)** : 625M$ voles - attaque sur les validators\n", "- **Wormhole (2022)** : 320M$ voles - bug de signature forgeable\n", "- **Harmony Bridge (2022)** : 100M$ voles - compromission des cles privees\n", "\n", "> **Note technique** : La regle d'or des bridges : \"Trust, but verify\". Faire confiance au bridge ne suffit pas, il faut verifier la finalite, la signature, et l'etat de la chaine source.\n"]

---

[<< Solana & Anchor](../05-Alternative-Chains/SC-22-Solana-Anchor.ipynb) | [Testnet Deploy >>](SC-24-Testnet-Deploy.ipynb)

["### Indice : Exercice Bridge Token Cross-Chain\n", "\n", "Pour implementer un bridge securise, rappelez-vous les proprietes critiques d'un systeme cross-chain :\n", "\n", "**Pattern Lock & Mint** :\n", "1. **Source chain** : L'utilisateur appelle `lock(amount)` → les tokens sont transfères dans le contrat bridge (verrouilles)\n", "2. **Detection** : Un oracle/relayer detecte l'evenement `Lock` et prouve la transaction sur la destination\n", "3. **Destination chain** : Le bridge appelle `mint(user, amount)` sur le `WrappedToken` → creation de tokens representatifs\n", "\n", "**Securite critique - Protection contre replay attacks** :\n", "- Le `nonce` doit etre unique et incrementer a chaque operation (timestamp ou compteur)\n", "- La signature doit inclure : user, amount, nonce, chainId (pour eviter la reutilisation sur une autre chaine)\n", "- `validateSignature` doit verifier que le signataire est le validator autorise (pas d'utilisateur arbitraire)\n", "\n", "**Indice implementation** :\n", "- Utiliser `ECDSA.recover(hash, signature)` pour verifier le signataire\n", "- Le hash a signer doit etre : `keccak256(abi.encodePacked(user, amount, nonce, chainId))`\n", "- Stocker les nonces utilises dans un `mapping(uint256 => bool)` pour empecher la reutilisation\n", "- `unlock` ne doit etre appelable que par le validator (pas par l'utilisateur directement)\n", "\n", "**Cas d'erreur a gerer** :\n", "- Le bridge destination est en pause (utilise `whenNotPaused`)\n", "- La signature est invalide (revert)\n", "- Le nonce a deja ete utilise (revert)\n", "- Le montant depasse la limite quotidienne (rate limiting)\n"]